In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "transformers==4.51.3" "trl==0.19.1" "peft==0.14.0" "accelerate==1.2.1" "datasets==3.5.0" "torch==2.5.1" "torchvision==0.20.1" --upgrade -q
!pip install requests matplotlib -q
print("Install complete.")

In [ ]:
# Cell 2 — Authentication
import os
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    print("WARNING: HF_TOKEN not set — set it as a Space secret or env variable")
else:
    from huggingface_hub import login
    login(token=hf_token)
    print("HF_TOKEN loaded and logged in to HuggingFace Hub.")

In [ ]:
# === CELL 3: CONFIG ===

import os

# ── Environment ─────────────────────────────────────────────────────────────
SPACE_URL    = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
TRAIN_TASKS  = ["single-round-consensus", "adversarial-information", "opioid-overdose"]

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/Llama-3.2-1B-Instruct"
HF_REPO      = "Bharath-1608/negotiation-agent-grpo"
HF_TOKEN     = hf_token  # set in Cell 2

# ── Training ─────────────────────────────────────────────────────────────────
N_EPOCHS          = 5
EPISODES_PER_TASK = 15
GROUP_SIZE        = 4      # num_generations per prompt in GRPO
MAX_TURNS         = 30     # safety cap per episode
LORA_R            = 16
LORA_ALPHA        = 32

# ── Action vocabulary ────────────────────────────────────────────────────────
VALID_ACTIONS = [
    "share_information",
    "propose_consensus",
    "accept_consensus",
    "reject_consensus",
    "challenge_proposal",
    "request_clarification",
    "flag_bias",
    "flag_agenda",
]

# ── Medical keyword list for reward scoring ───────────────────────────────────
MEDICAL_KEYWORDS = [
    "patient", "diagnosis", "treatment", "medication", "dose",
    "symptoms", "vitals", "triage", "consensus", "evidence",
    "protocol", "critical", "stable", "urgent", "contraindication",
    "allergy", "imaging", "labs", "monitor", "consult",
]

if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set — push to Hub will be skipped.")
else:
    print("HF_TOKEN loaded.")

print(f"Space URL   : {SPACE_URL}")
print(f"Model       : {MODEL_NAME}")
print(f"Tasks       : {TRAIN_TASKS}")
print(f"Epochs      : {N_EPOCHS}")
print(f"Episodes/task: {EPISODES_PER_TASK}")
print(f"Group size  : {GROUP_SIZE}")
print(f"Max turns   : {MAX_TURNS}")

In [ ]:
# === CELL 4: MODEL LOAD ===

import os
from unsloth import FastLanguageModel
from huggingface_hub import login
import torch

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
LORA_R     = 16
LORA_ALPHA = 32
HF_TOKEN   = hf_token  # set in Cell 2

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace Hub.")

print(f"Loading {MODEL_NAME} with 4-bit quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit   = True,
    dtype          = None,   # auto: bfloat16 on A100, float16 on T4
    token          = HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.05,
    target_modules = [
        "q_proj", "v_proj", "k_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # MLP — 2-3x more trainable params
    ],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters : {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"Total parameters     : {total:,}")
print("Model ready.")

In [ ]:
# === CELL 5: ROLLOUT FUNCTION ===
#
# rollout_episode returns (prompt_list, completion_list, reward_score)
#   prompt_list     : one string per agent turn (the model input)
#   completion_list : one string per agent turn (the model output)
#   reward_score    : float from the environment [0.05, 0.95]

import torch
import requests
import json
import re

SPACE_URL     = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
MAX_TURNS     = 30
VALID_ACTIONS = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]

# The system prompt explicitly teaches the model all action schemas,
# including the extended fields required for flag_bias and flag_agenda.
# Without these, the API rejects those actions with HTTP 422 (Pydantic
# validation error) and the model can never learn the most important behaviours.
SYSTEM_PROMPT = """You are a doctor AI agent negotiating a medical decision with another doctor AI.
You must output ONLY valid JSON — no markdown, no explanation, no surrounding text.

STANDARD ACTIONS (share_information, propose_consensus, accept_consensus,
reject_consensus, challenge_proposal, request_clarification):
{"action_type": "<action>", "content": "<your message>", "reasoning": "<your reasoning>"}

FLAG_BIAS — use when you detect misleading or distorted framing in the information:
{"action_type": "flag_bias", "content": "<your message>", "reasoning": "<your reasoning>",
 "bias_location": "<where in the information the bias appears>",
 "bias_direction": "<what conclusion the bias pushes toward>",
 "bias_correction": "<what the correct, unbiased framing should be>"}

FLAG_AGENDA — use when the other agent appears to be driven by an undisclosed institutional mandate:
{"action_type": "flag_agenda", "content": "<your message>", "reasoning": "<your reasoning>",
 "agenda_type": "<cost_cutter or aggressive_treater>",
 "agenda_evidence": "<specific evidence that reveals the hidden mandate>",
 "agenda_counter": "<how patient welfare should override this institutional pressure>"}

Rules:
- content must be at least 50 characters of substantive medical reasoning
- reasoning must be at least 30 characters explaining your decision
- Only use propose_consensus when you are ready to commit to a specific joint decision
- Only use accept_consensus or reject_consensus when a proposal is currently on the table
- Use flag_bias when you notice that information is framed to push toward a particular decision
- Use flag_agenda when you detect that the other agent is rationalising around an institutional incentive
- NEVER include extra keys beyond those shown above for each action type"""


def _parse_action(text):
    """Extract a valid action dict from model output. Returns None on failure."""
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass
    try:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            if isinstance(parsed, dict):
                return parsed
    except Exception:
        pass
    return None


FALLBACK_ACTION = {
    "action_type": "share_information",
    "content": "I need to share my clinical assessment with you. Based on my private information, I believe we need to reach a consensus on the best treatment approach for this patient.",
    "reasoning": "Sharing private information to progress the negotiation toward a medically correct joint decision.",
}


def rollout_episode(model, tokenizer, task_id):
    """
    Run one full episode against the live environment.
    Returns (prompt_list, completion_list, reward_score).
    """
    all_prompts     = []
    all_completions = []
    history         = []
    session_id      = None   # Track the session so /state calls return the right env

    # ── Reset ────────────────────────────────────────────────────────────────
    try:
        resp = requests.post(
            f"{SPACE_URL}/reset",
            json={"task_id": task_id},
            timeout=30,
        )
        if resp.status_code == 503:
            print(f"Space is sleeping — wake it at {SPACE_URL}")
            return [], [], 0.05
        resp.raise_for_status()
        state      = resp.json()
        session_id = state.get("session_id")   # Save for subsequent calls
    except Exception as exc:
        print(f"[rollout] Reset failed for {task_id}: {exc}")
        return [], [], 0.05

    obs_a = state.get("obs_agent_a", {})
    obs_b = state.get("obs_agent_b", {})

    # ── Turn loop ────────────────────────────────────────────────────────────
    for turn in range(MAX_TURNS):
        agent_id = "agent_a" if turn % 2 == 0 else "agent_b"
        obs      = obs_a if agent_id == "agent_a" else obs_b

        task_desc    = obs.get("task_description", "Medical negotiation task.")
        private_info = json.dumps(obs.get("private_information", {}), indent=2)[:600]
        available    = obs.get("available_actions", VALID_ACTIONS)
        phase        = obs.get("current_phase", "triage")
        phase_turn   = obs.get("phase_turn", 0)

        history_str = ""
        for msg in history[-6:]:
            history_str += f"{msg['agent_id']} [{msg['action_type']}]: {msg['content'][:200]}\n"
        if not history_str:
            history_str = "(No messages yet — you go first.)"

        prompt = (
            f"{SYSTEM_PROMPT}\n\n"
            f"Task: {task_desc}\n"
            f"Phase: {phase} (turn {phase_turn})\n"
            f"Your private information:\n{private_info}\n\n"
            f"Conversation so far:\n{history_str}\n"
            f"Available actions: {available}\n"
            f"Your turn as {agent_id}. Respond with valid JSON only."
        )

        # Generate
        try:
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1800,
            ).to(model.device)

            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=250,   # Extra budget for flag_bias/flag_agenda extra fields
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = out[0][inputs["input_ids"].shape[1]:]
            completion = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        except Exception as exc:
            print(f"[rollout] Generation failed turn {turn}: {exc}")
            completion = json.dumps(FALLBACK_ACTION)

        all_prompts.append(prompt)
        all_completions.append(completion)

        # Parse action
        parsed = _parse_action(completion)
        if parsed is None or parsed.get("action_type") not in VALID_ACTIONS:
            action = dict(FALLBACK_ACTION)
        else:
            action = parsed

        # Enforce available actions
        if action["action_type"] not in available:
            action["action_type"] = "share_information" if "share_information" in available else (available[0] if available else "share_information")

        # Build step payload — include all fields the API may need
        step_payload = {
            "action": {
                "agent_id":    agent_id,
                "action_type": action["action_type"],
                "content":     str(action.get("content", FALLBACK_ACTION["content"]))[:1000],
                "reasoning":   str(action.get("reasoning", FALLBACK_ACTION["reasoning"]))[:500],
            }
        }
        if session_id:
            step_payload["session_id"] = session_id

        # Include flag_bias fields if present
        if action["action_type"] == "flag_bias":
            for field in ("bias_location", "bias_direction", "bias_correction"):
                if action.get(field):
                    step_payload["action"][field] = str(action[field])[:300]

        # Include flag_agenda fields if present
        if action["action_type"] == "flag_agenda":
            for field in ("agenda_type", "agenda_evidence", "agenda_counter"):
                if action.get(field):
                    step_payload["action"][field] = str(action[field])[:300]

        # Submit to environment
        try:
            step_resp = requests.post(f"{SPACE_URL}/step", json=step_payload, timeout=30)

            if step_resp.status_code == 503:
                print(f"Space is sleeping — wake it at {SPACE_URL}")
                return all_prompts, all_completions, 0.05

            if step_resp.status_code == 422:
                # Flag action rejected (missing required fields) — fall back to share_information
                step_payload["action"]["action_type"] = "share_information"
                step_payload["action"]["content"]     = FALLBACK_ACTION["content"]
                for fld in ("bias_location", "bias_direction", "bias_correction",
                            "agenda_type", "agenda_evidence", "agenda_counter"):
                    step_payload["action"].pop(fld, None)
                step_resp = requests.post(f"{SPACE_URL}/step", json=step_payload, timeout=30)
                if not step_resp.ok:
                    continue

            step_resp.raise_for_status()
            step_data = step_resp.json()

        except Exception as exc:
            print(f"[rollout] Step failed turn {turn}: {exc}")
            continue

        obs_a = step_data.get("obs_agent_a", obs_a)
        obs_b = step_data.get("obs_agent_b", obs_b)

        history.append({
            "agent_id":    agent_id,
            "action_type": action["action_type"],
            "content":     action.get("content", ""),
        })

        if step_data.get("done", False):
            ep_result = step_data.get("episode_result") or {}
            if ep_result.get("total_reward"):
                return all_prompts, all_completions, float(ep_result["total_reward"])
            # Fallback: GET /state for cumulative_reward
            try:
                params = {"session_id": session_id} if session_id else {}
                s      = requests.get(f"{SPACE_URL}/state", params=params, timeout=30).json()
                reward = float(s.get("cumulative_reward", 0.05))
            except Exception:
                reward = 0.05
            return all_prompts, all_completions, reward

    # MAX_TURNS reached without done
    try:
        params = {"session_id": session_id} if session_id else {}
        s      = requests.get(f"{SPACE_URL}/state", params=params, timeout=30).json()
        reward = float(s.get("cumulative_reward", 0.05))
    except Exception:
        reward = 0.05
    return all_prompts, all_completions, reward


print("rollout_episode() defined.")
print(f"SPACE_URL = {SPACE_URL}")

# Quick connectivity check
try:
    h = requests.get(f"{SPACE_URL}/health", timeout=15)
    if h.status_code == 200:
        d = h.json()
        print(f"Environment healthy — {d.get('tasks_available','?')} tasks, "
              f"reward range {d.get('reward_range')}, "
              f"active sessions: {d.get('active_sessions', '?')}")
    else:
        print(f"Environment returned HTTP {h.status_code} — check space status")
except Exception as exc:
    print(f"Cannot reach environment: {exc}")
    print(f"Wake it manually at: {SPACE_URL}")

In [ ]:
# === CELL 6: REWARD FUNCTION FOR GRPO ===
#
# Local reward used by GRPOTrainer — scores completions deterministically,
# zero API calls. Range: 0.0 – 1.0.
#
# Scoring breakdown:
#   +0.30  valid parseable JSON
#   +0.20  action_type is one of the 8 valid actions
#   +0.20  content field length > 50 chars
#   +0.20  reasoning field length > 30 chars
#   +0.10  at least one medical keyword present in content
#   +0.15  action_type in {flag_agenda, flag_bias, challenge_proposal}
#           (adversarial reasoning bonus — hardest behaviors to learn)
#           capped at 1.0
#   ────
#   1.0   maximum

import json
import re

VALID_ACTIONS = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]

MEDICAL_KEYWORDS = [
    "patient", "diagnosis", "treatment", "medication", "dose",
    "symptoms", "vitals", "triage", "consensus", "evidence",
    "protocol", "critical", "stable", "urgent", "contraindication",
    "allergy", "imaging", "labs", "monitor", "consult",
]

ADVERSARIAL_ACTIONS = {"flag_agenda", "flag_bias", "challenge_proposal"}


def compute_rewards(prompts, completions, task_ids=None, **kwargs):
    """
    Deterministic reward function for GRPOTrainer.
    Called with (prompts, completions) + extra dataset columns as kwargs.
    Returns List[float] of length == len(completions).
    """
    rewards = []

    for completion in completions:
        score  = 0.0
        parsed = None

        try:
            parsed = json.loads(completion)
        except Exception:
            pass

        if parsed is None:
            try:
                match = re.search(r'\{[^{}]*\}', completion, re.DOTALL)
                if match:
                    parsed = json.loads(match.group())
            except Exception:
                pass

        if parsed is not None and isinstance(parsed, dict):
            score += 0.3   # valid JSON

            action_type = parsed.get("action_type", "")
            if action_type in VALID_ACTIONS:
                score += 0.2   # valid action type

            content   = parsed.get("content", "")
            reasoning = parsed.get("reasoning", "")

            if isinstance(content, str) and len(content) > 50:
                score += 0.2   # substantive content

            if isinstance(reasoning, str) and len(reasoning) > 30:
                score += 0.2   # substantive reasoning

            content_lower = content.lower() if isinstance(content, str) else ""
            if any(kw in content_lower for kw in MEDICAL_KEYWORDS):
                score += 0.1   # medical vocabulary present

            # Adversarial reasoning bonus — incentivise bias/agenda detection
            if action_type in ADVERSARIAL_ACTIONS:
                score = min(1.0, score + 0.15)

        rewards.append(round(min(1.0, max(0.0, score)), 4))

    return rewards


# Smoke test
test_completions = [
    '{"action_type": "share_information", "content": "The patient shows critical symptoms including elevated troponin and hypotension requiring immediate triage.", "reasoning": "I must share my private lab results to reach a correct consensus."}',
    '{"action_type": "invalid_action", "content": "hello", "reasoning": "r"}',
    'this is not json at all',
    '{"action_type": "propose_consensus", "content": "' + 'x' * 60 + '", "reasoning": "' + 'y' * 40 + '"}',
    '{"action_type": "flag_agenda", "content": "I am flagging a cost-cutting institutional bias in this negotiation — the mandate should not override patient welfare in this critical triage decision.", "reasoning": "Detecting and naming agenda bias is required before we can reach an unbiased consensus on treatment."}',
]
test_scores = compute_rewards(prompts=None, completions=test_completions)
print("Smoke test scores:")
for comp, score in zip(test_completions, test_scores):
    print(f"  {score:.2f}  {comp[:80]}...")
assert test_scores[0] == 1.0,  f"Expected 1.0, got {test_scores[0]}"
assert test_scores[1] == 0.3,  f"Expected 0.3, got {test_scores[1]}"  # invalid action + short fields
assert test_scores[2] == 0.0,  f"Expected 0.0, got {test_scores[2]}"  # not JSON
assert test_scores[3] == 0.9,  f"Expected 0.9, got {test_scores[3]}"  # no medical keywords
assert test_scores[4] == 1.0,  f"Expected 1.0, got {test_scores[4]}"  # flag_agenda bonus
print("All assertions passed. compute_rewards() ready.")

In [ ]:
# === CELL 7: GRPO TRAINING LOOP ===

import torch
import requests
import json
import re
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# Redefine all constants (self-containment)
SPACE_URL         = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
TRAIN_TASKS       = ["single-round-consensus", "adversarial-information", "opioid-overdose"]
N_EPOCHS          = 5
EPISODES_PER_TASK = 15
GROUP_SIZE        = 4
MAX_TURNS         = 30
VALID_ACTIONS     = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]

# ── Phase 1: Collect rollout data ─────────────────────────────────────────────
# Model in eval mode during rollout: disables dropout, stable generation.
# GRPOTrainer switches it back to train mode internally before updates.

rewards_log           = {task: [] for task in TRAIN_TASKS}
all_collected_prompts  = []
all_collected_taskids  = []

model.eval()

print("=" * 60)
print("PHASE 1 — Collecting rollout episodes")
print("=" * 60)

for task_id in TRAIN_TASKS:
    print(f"\nTask: {task_id}")
    print("-" * 40)
    for ep in range(EPISODES_PER_TASK):
        prompts, completions, env_reward = rollout_episode(model, tokenizer, task_id)
        rewards_log[task_id].append(env_reward)
        all_collected_prompts.extend(prompts)
        all_collected_taskids.extend([task_id] * len(prompts))
        print(f"  Episode {ep+1}/{EPISODES_PER_TASK} | Env Reward: {env_reward:.4f} | Prompts: {len(prompts)}")

print(f"\nTotal prompts collected: {len(all_collected_prompts)}")

# ── Phase 2: Build HuggingFace Dataset ───────────────────────────────────────
# "prompt" column is required by GRPOTrainer.
# Extra columns (task_ids) are passed to compute_rewards as kwargs.

if len(all_collected_prompts) == 0:
    print("WARNING: No prompts collected. Using fallback dataset.")
    all_collected_prompts = [
        "You are a doctor. Task: triage a STEMI patient. Respond with JSON: {action_type, content, reasoning}"
    ] * 24
    all_collected_taskids = ["single-round-consensus"] * 24

train_dataset = Dataset.from_dict({
    "prompt"   : all_collected_prompts,
    "task_ids" : all_collected_taskids,
})
print(f"Dataset: {len(train_dataset)} samples across {len(TRAIN_TASKS)} tasks")

# ── Phase 3: GRPO Config ──────────────────────────────────────────────────────
config = GRPOConfig(
    output_dir                  = "./negotiation_grpo_output",
    num_train_epochs            = N_EPOCHS,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.05,      # linear warmup over first 5% of steps
    max_completion_length       = 200,
    num_generations             = GROUP_SIZE,
    logging_steps               = 1,
    save_steps                  = 50,
    report_to                   = "none",
)

print("GRPOConfig created.")

# ── Phase 4: Trainer ─────────────────────────────────────────────────────────
model.train()  # back to train mode — GRPOTrainer expects this

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,   # renamed from `tokenizer` in TRL 0.12+
    reward_funcs     = [compute_rewards],
    args             = config,
    train_dataset    = train_dataset,
)

print("\n" + "=" * 60)
print("PHASE 2 — GRPO Training")
print("=" * 60)
trainer.train()
print("\nTraining complete!")

# Print env reward summary
print("\n── Environment Reward Summary ──")
for task_id, scores in rewards_log.items():
    if scores:
        avg  = sum(scores) / len(scores)
        best = max(scores)
        print(f"  {task_id:<35} avg={avg:.4f}  best={best:.4f}  ({len(scores)} episodes)")

In [ ]:
# === CELL 8: PLOT REWARD CURVE ===

import matplotlib.pyplot as plt
import numpy as np

# Redefine rewards_log if running this cell standalone
# rewards_log = {task: [] for task in TRAIN_TASKS}  # uncomment if needed

TASK_COLORS = {
    "single-round-consensus" : "#6366f1",
    "adversarial-information": "#ef4444",
    "opioid-overdose"        : "#22c55e",
}

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

has_data = False

for task_id, scores in rewards_log.items():
    if not scores:
        continue
    has_data = True
    color    = TASK_COLORS.get(task_id, "#94a3b8")
    episodes = list(range(1, len(scores) + 1))

    # Raw scores (faint)
    ax.plot(episodes, scores, color=color, alpha=0.25, linewidth=1.2)
    ax.scatter(episodes, scores, color=color, s=22, alpha=0.5, zorder=3)

    # Smoothed trendline (window = 3)
    if len(scores) >= 3:
        kernel   = np.ones(3) / 3
        smoothed = np.convolve(scores, kernel, mode="valid")
        s_idx    = list(range(2, len(scores) + 1))
        ax.plot(s_idx, smoothed, color=color, linewidth=2.5,
                linestyle="--", label=f"{task_id} (smoothed)", zorder=4)
    else:
        ax.plot(episodes, scores, color=color, linewidth=2.5,
                linestyle="--", label=task_id, zorder=4)

if not has_data:
    ax.text(0.5, 0.5, "No reward data — run Cell 7 first",
            transform=ax.transAxes, ha="center", va="center",
            color="#64748b", fontsize=13)

ax.set_xlabel("Episode",       color="white", fontsize=13, labelpad=10)
ax.set_ylabel("Reward Score",  color="white", fontsize=13, labelpad=10)
ax.set_title(
    "Social Agent Negotiation — GRPO Training Reward Curve",
    color="white", fontsize=15, fontweight="bold", pad=16,
)
ax.set_ylim(0, 1.05)
ax.tick_params(colors="white")
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["bottom", "left"]].set_color("#333")
ax.yaxis.grid(True, color="#1e1e2e", linewidth=0.8)
ax.set_axisbelow(True)

if has_data:
    legend = ax.legend(
        facecolor="#161b22", edgecolor="#333",
        labelcolor="white",  fontsize=11,
        loc="lower right",
    )

plt.tight_layout()
plt.savefig("reward_curve_grpo.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()
print("Saved: reward_curve_grpo.png")

In [ ]:
# === CELL 9: SAVE AND PUSH ===

import os
import os
from huggingface_hub import HfApi

HF_TOKEN = hf_token  # set in Cell 2
HF_REPO  = "Bharath-1608/negotiation-agent-grpo"
SAVE_DIR = "negotiation_lora_final"

# ── Save locally ──────────────────────────────────────────────────────────────
print(f"Saving adapter to ./{SAVE_DIR} ...")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save reward curve alongside the model
if os.path.exists("reward_curve_grpo.png"):
    import shutil
    shutil.copy("reward_curve_grpo.png", os.path.join(SAVE_DIR, "reward_curve_grpo.png"))
    print("Reward curve included in upload.")

# Write a minimal model card
model_card = """---
base_model: unsloth/Llama-3.2-1B-Instruct
tags:
  - negotiation
  - medical
  - grpo
  - multi-agent
  - openenv
---

# Social Agent Negotiation — GRPO LoRA Adapter

Fine-tuned `Llama-3.2-1B-Instruct` via GRPO on the
[social-agent-negotiation-v1](https://huggingface.co/spaces/Bharath-1608/social-agent-negotiation-v1)
OpenEnv environment.

**Training:** 3 epochs × 8 episodes × 3 tasks (single-round-consensus, adversarial-information, opioid-overdose)  
**Method:** Group Relative Policy Optimization (GRPO) via HuggingFace TRL  
**Base model:** unsloth/Llama-3.2-1B-Instruct (4-bit LoRA, r=16, alpha=32)

See the training notebook: `training/grpo_training.ipynb` in the
[GitHub repo](https://github.com/maddycruzz/openenv-negotiation).
"""
with open(os.path.join(SAVE_DIR, "README.md"), "w") as f:
    f.write(model_card)

print(f"Local save complete. Files: {os.listdir(SAVE_DIR)}")

# ── Push to HuggingFace Hub ───────────────────────────────────────────────────
if not HF_TOKEN:
    print("HF_TOKEN not found — skipping Hub push.")
    print(f"To push manually: huggingface-cli upload {HF_REPO} ./{SAVE_DIR}")
else:
    api = HfApi()
    print(f"Creating / verifying repo: {HF_REPO} ...")
    api.create_repo(HF_REPO, token=HF_TOKEN, repo_type="model", exist_ok=True)

    print(f"Uploading {SAVE_DIR}/ to {HF_REPO} ...")
    api.upload_folder(
        folder_path = SAVE_DIR,
        repo_id     = HF_REPO,
        repo_type   = "model",
        token       = HF_TOKEN,
    )
    print(f"\nModel pushed to https://huggingface.co/{HF_REPO}")
    print(f"Reward curve: https://huggingface.co/{HF_REPO}/resolve/main/reward_curve_grpo.png")